# EyePACS patient-level holdout

Splits the EyePACS training images into train and a 5% holdout for stage-1 early stopping.

- **By patient:** both eyes (`<id>_left`, `<id>_right`) always land in the same split, so near-identical retinas can't leak between them.
- **Stratified by the patient's worse eye**, so every grade (including the rare 3s and 4s) shows up in the holdout in about the same proportion as in train.
- **Fixed seed**, so re-running gives the same split.

Writes `splits/eyepacs_train.csv` and `splits/eyepacs_holdout.csv` with columns `image, level, patient, path`.

In [1]:
import random
from collections import defaultdict
from pathlib import Path

import pandas as pd

HOLDOUT_FRAC = 0.05
SEED = 42
LABELS = Path("data/trainLabels.csv")
IMAGES = Path("data/processed/train/train")
SPLITS = Path("splits")

df = pd.read_csv(LABELS)
df["patient"] = df["image"].str.rsplit("_", n=1).str[0]
df["path"] = [str(IMAGES / f"{i}.png") for i in df["image"]]
missing = [p for p in df["path"] if not Path(p).exists()]
assert not missing, f"{len(missing)} labelled images have no processed PNG, e.g. {missing[:3]}"

# Stratify on each patient's worse eye.
patient_grade = df.groupby("patient")["level"].max()
by_grade = defaultdict(list)
for patient, grade in patient_grade.items():
    by_grade[grade].append(patient)

rng = random.Random(SEED)
holdout_patients = set()
for grade, patients in sorted(by_grade.items()):
    patients = sorted(patients)
    rng.shuffle(patients)
    holdout_patients.update(patients[:round(len(patients) * HOLDOUT_FRAC)])

is_holdout = df["patient"].isin(holdout_patients)
train, holdout = df[~is_holdout], df[is_holdout]
assert not set(train["patient"]) & set(holdout["patient"])

SPLITS.mkdir(exist_ok=True)
train.to_csv(SPLITS / "eyepacs_train.csv", index=False)
holdout.to_csv(SPLITS / "eyepacs_holdout.csv", index=False)

summary = pd.DataFrame({
    "train": train["level"].value_counts().sort_index(),
    "holdout": holdout["level"].value_counts().sort_index(),
})
summary["holdout %"] = (100 * summary["holdout"] / (summary["train"] + summary["holdout"])).round(1)
print(f"patients: {train['patient'].nunique()} train, {holdout['patient'].nunique()} holdout")
print(f"images:   {len(train)} train, {len(holdout)} holdout")
summary

patients: 16685 train, 878 holdout
images:   33370 train, 1756 holdout


,train,holdout,holdout %
level,,,
0,24526,1284,5.0
1,2317,126,5.2
2,5026,266,5.0
3,828,45,5.2
4,673,35,4.9


## APTOS 2019 (existing split)

Uses the labelled train / valid / test split from the `mariaherrerot/aptos2019` Kaggle dataset, which is the same split as the earlier APTOS work, and only rewrites it with paths to the processed 512×512 PNGs.

Writes `splits/aptos_train.csv`, `splits/aptos_dev.csv` and `splits/aptos_test.csv` (columns `image, level, path`). **`aptos_test.csv` stays unused until final evaluation.**

In [2]:
APTOS_CSVS = Path("data/aptos2019")
APTOS_IMAGES = Path("data/processed/aptos2019")
aptos_splits = {                  # our name: (label CSV, processed image folder)
    "train": ("train_1.csv", "train_images/train_images"),
    "dev": ("valid.csv", "val_images/val_images"),
    "test": ("test.csv", "test_images/test_images"),
}

aptos = {}
for split, (csv, folder) in aptos_splits.items():
    d = pd.read_csv(APTOS_CSVS / csv).rename(columns={"id_code": "image", "diagnosis": "level"})
    d["path"] = [str(APTOS_IMAGES / folder / f"{i}.png") for i in d["image"]]
    missing = [p for p in d["path"] if not Path(p).exists()]
    assert not missing, f"{split}: {len(missing)} images missing, e.g. {missing[:3]}"
    d.to_csv(SPLITS / f"aptos_{split}.csv", index=False)
    aptos[split] = d

ids = {s: set(d["image"]) for s, d in aptos.items()}
assert not (ids["train"] & ids["dev"] or ids["train"] & ids["test"] or ids["dev"] & ids["test"])
pd.DataFrame({s: d["level"].value_counts().sort_index() for s, d in aptos.items()})

,train,dev,test
level,,,
0,1434,172,199
1,300,40,30
2,808,104,87
3,154,22,17
4,234,28,33
